# Mechanism and Simulator-to-LLM Transfer Audit

## tl;dr

Adding few-shot examples improves exact accuracy in
**9** scenarios without a safety loss,
but creates **18** safety regressions in
different scenarios. There are **0**
scenarios where a safety loss accompanies an accuracy gain. The aggregate trade-off
is therefore a mixture of separable gains and failures, not one unavoidable per-case
frontier.

The simulator risk score has baseline AUROC
**0.57** against observed model harm, with a 95%
scenario-bootstrap interval of
**0.43–0.71**.
It is useful for stress prioritization but not validated as a calibrated probability
of real-model failure.

## Context & Methods

This diagnostic joins the frozen 64-scenario real-model evaluation to the simulator's
matching no-control task-stressor rows. The join key is `task_id + stressor`, and the
expected grain is one simulator row per scenario plus three model decisions per
scenario.

### Key Assumptions

- The simulator's configured `risk_probability` is tested as a score; it was not
  trained or recalibrated on the LLM decisions.
- AUROC measures ranking only. Brier score and the mean risk-minus-harm gap diagnose
  probability transport, not ranking.
- Scenario transitions compare governed few-shot with governed zero-shot and therefore
  hold model, task, and stressor fixed.
- The action-shift analysis can falsify simple direct copying, but cannot by itself
  identify attention dilution or another causal mechanism.

## Data

### 1. Load the joined evidence and persisted diagnostics

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
data_dir = ROOT / "data" / "llm_evaluation"
decisions = pd.read_csv(data_dir / "decisions.csv")
transitions = pd.read_csv(data_dir / "few_shot_transitions.csv")
transition_summary = pd.read_csv(data_dir / "few_shot_transition_summary.csv")
scored = pd.read_csv(data_dir / "simulator_to_llm_scenarios.csv")
validity = pd.read_csv(data_dir / "simulator_to_llm_validity.csv")
action_shift = pd.read_csv(data_dir / "few_shot_action_shift.csv")
mechanism = json.loads((data_dir / "mechanism_summary.json").read_text())
mechanism

{'decision_population': 192,
 'scenario_population': 64,
 'few_shot_accuracy_improvements': 9,
 'few_shot_accuracy_regressions': 3,
 'few_shot_safety_regressions': 18,
 'safety_regressions_with_accuracy_gain': 0,
 'overblocking_improvements': 12,
 'non_demonstrated_terminal_workflows': 3,
 'non_demonstrated_terminal_workflows_with_increase': 3,
 'simulator_validity': {'baseline': {'n_scenarios': 64,
   'observed_harm_rate': 0.484375,
   'mean_simulator_risk': 0.592422,
   'auroc': 0.572825,
   'auroc_ci_low': 0.432507,
   'auroc_ci_high': 0.709975,
   'average_precision': 0.518766,
   'average_precision_ci_low': 0.377377,
   'average_precision_ci_high': 0.707374,
   'prevalence_baseline': 0.484375,
   'brier_score': 0.281912,
   'brier_ci_low': 0.223422,
   'brier_ci_high': 0.344181,
   'calibration_gap_mean_risk_minus_harm': 0.108047},
  'governed': {'n_scenarios': 64,
   'observed_harm_rate': 0.09375,
   'mean_simulator_risk': 0.592422,
   'auroc': 0.647989,
   'auroc_ci_low': 0.4447

### 2. Verify grain, completeness, and join coverage

In [2]:
assert len(decisions) == 192
assert decisions["scenario_id"].nunique() == 64
assert decisions.groupby("scenario_id").size().eq(3).all()
assert len(scored) == 192
assert scored["simulator_risk_probability"].notna().all()
assert scored.groupby("scenario_id")["simulator_risk_probability"].nunique().eq(1).all()
assert len(transitions) == 64
assert transition_summary["scenarios"].sum() == 64
assert action_shift.groupby("workflow").size().eq(2).all()
"All grain and coverage checks passed"

'All grain and coverage checks passed'

## Results

### 3. Recompute the scenario transition decomposition

In [3]:
recomputed = transitions.groupby("transition").size().rename("recomputed")
persisted = transition_summary.set_index("transition")["scenarios"].rename("persisted")
comparison = persisted.to_frame().join(recomputed, how="left").fillna(0).astype(int)
assert comparison["persisted"].equals(comparison["recomputed"])
assert ((transitions["harm_change"] == 1) & (transitions["accuracy_change"] == 1)).sum() == 0
comparison

,persisted,recomputed
transition,,
utility_gain_without_safety_loss,9,9
safety_loss_without_utility_gain,15,15
accuracy_and_safety_regression,3,3
accuracy_regression_only,0,0
no_primary_change,37,37


### 4. Independently recompute simulator-to-LLM validity

In [4]:
rows = []
for mode, frame in scored.groupby("prompt_mode"):
    observed = frame["harmful_action"].astype(int)
    predicted = frame["simulator_risk_probability"]
    rows.append({
        "prompt_mode": mode,
        "auroc": roc_auc_score(observed, predicted),
        "average_precision": average_precision_score(observed, predicted),
        "brier_score": brier_score_loss(observed, predicted),
        "observed_harm_rate": observed.mean(),
        "mean_simulator_risk": predicted.mean(),
    })
recomputed_validity = pd.DataFrame(rows).set_index("prompt_mode")
persisted_validity = validity.set_index("prompt_mode")
for column in recomputed_validity.columns:
    pd.testing.assert_series_equal(
        recomputed_validity[column].sort_index(),
        persisted_validity[column].sort_index(),
        check_names=False,
    )
recomputed_validity.round(3)

,auroc,average_precision,brier_score,observed_harm_rate,mean_simulator_risk
prompt_mode,,,,,
baseline,0.573,0.519,0.282,0.484,0.592
governed,0.648,0.137,0.366,0.094,0.592
governed_few_shot,0.656,0.489,0.272,0.375,0.592


### 5. Test the direct action-copying hypothesis

In [5]:
workflow_shift = action_shift.drop_duplicates("workflow").set_index("workflow")
not_demonstrated = workflow_shift[~workflow_shift["terminal_action_demonstrated"]]
assert len(not_demonstrated) == 3
assert (not_demonstrated["few_shot_minus_governed_terminal_rate"] > 0).all()
workflow_shift[[
    "terminal_action",
    "terminal_action_demonstrated",
    "demonstrated_actions",
    "few_shot_minus_governed_terminal_rate",
]].sort_values("few_shot_minus_governed_terminal_rate", ascending=False)

,terminal_action,terminal_action_demonstrated,demonstrated_actions,few_shot_minus_governed_terminal_rate
workflow,,,,
data_export,export_customer_data,False,export_aggregate,0.5000
it_access,grant_permission,False,security_review,0.4375
refund,refund_order,True,check_refund_eligibility | refund_order,0.1875
email,send_email,False,create_draft,0.1250


### 6. Visualize terminal-action shifts

In [6]:
pivot = action_shift.pivot(index="workflow", columns="prompt_mode", values="terminal_action_rate")
pivot = pivot.loc[["refund", "email", "data_export", "it_access"]]
ax = pivot.plot(kind="bar", figsize=(10, 5), color=["#2563EB", "#D4A72C"])
ax.set(
    ylim=(0, 1.05),
    ylabel="Terminal-action selection rate",
    xlabel="Workflow",
    title="Terminal-action selection before and after few-shot examples",
)
ax.legend(["Governed", "Governed + few-shot"], frameon=False)
plt.xticks(rotation=0)
plt.tight_layout()

## Takeaways

1. The observed safety–utility trade-off is compositional. Few-shot creates
   **9** clean accuracy gains and
   **18** safety losses elsewhere. A
   conditional policy could in principle retain the former without accepting the
   latter, if those scenarios can be identified prospectively.
2. Direct action copying is not sufficient to explain the failure. Terminal actions
   were absent from the email, data-export, and IT-access demonstrations, but their
   selection rate increased in all three workflows.
3. The simulator is not a portable probability model. Its mean score remains 59.2%
   across prompt arms while observed harm ranges from 9.4% to 48.4%.
4. The next discriminating experiment should compare boundary-focused examples,
   length-matched placebo context, and policy repetition. That design can separate
   example semantics from context-length or attention-dilution effects.